# Lab 02_2. 신경망 형식의 선형회귀

## 목표

- `Lab 02_1`의 직접 계산 구조를 PyTorch(파이토치) 신경망 형식으로 바꾸기
- `nn.Module`(엔엔 모듈, 신경망 모델의 기본 클래스)로 모델 정의하기
- `nn.Linear`(엔엔 리니어, 선형 계층)의 가중치와 편향 이해하기
- `nn.MSELoss()`(엔엔 엠에스이 로스, 평균제곱오차 손실함수) 사용하기
- 기존 학습 반복문에서 변경되는 부분 구분하기


## 실습 단계

### 1. 라이브러리 불러오기


In [ ]:
import torch
from torch import nn

# 재현성: 난수 고정
torch.manual_seed(42)

print("PyTorch 버전:", torch.__version__)


### 2. 학습 데이터 준비하기

`nn.Linear(1, 1)`은 한 행에 입력 특성 1개를 받습니다.  
따라서 `Lab 02_1`의 1차원 텐서를 `[데이터 수, 입력 특성 수]` 형상으로 바꿉니다.

- `Lab 02_1`: `x`, `y` 형상 → `[3]`
- `Lab 02_2`: `x`, `y` 형상 → `[3, 1]`


In [ ]:
# Lab 02_1: 1차원 텐서
# x_train = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)
# y_train = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)

# (변경) 입력 형상: [데이터 수, 특성 수]
x_train = torch.tensor([[1.0], [2.0], [3.0]], dtype=torch.float32)
y_train = torch.tensor([[1.0], [2.0], [3.0]], dtype=torch.float32)

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)


### 3. 모델 정의하기

`Lab 02_1`에서는 가중치 $W$와 편향 $b$를 직접 만들었습니다.  
`Lab 02_2`에서는 `nn.Linear(1, 1)`이 $W$와 $b$를 내부에 생성하고 관리합니다.

$$
\hat{y}=Wx+b
$$

ReLU(렐루, 음수를 0으로 바꾸는 활성화 함수)는 사용하지 않습니다.  
이번 단계의 목적은 같은 선형회귀를 신경망 형식으로 바꾸는 것입니다.


In [ ]:
# Lab 02_1: W·b 직접 생성
# W = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)
# b = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)


# (변경) 신경망 모델 정의
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()

        # 선형 계층: 입력 1개 → 출력 1개
        # 학습 대상: W·b
        self.linear = nn.Linear(in_features=1, out_features=1)

    def forward(self, x):
        # 순전파: y_hat = W*x + b
        prediction = self.linear(x)
        return prediction


model = LinearRegressionModel()
print(model)


### 4. 모델 안의 가중치와 편향 확인하기

`model.parameters()`를 사용하면 모델 내부의 학습 가능한 매개변수를 한꺼번에 가져올 수 있습니다.


In [ ]:
print("초기 W:", model.linear.weight.item())
print("초기 b:", model.linear.bias.item())

for name, parameter in model.named_parameters():
    print(name, parameter.shape, parameter.requires_grad)


### 5. 손실함수와 최적화 도구 바꾸기

직접 작성했던 MSE(엠에스이, 평균제곱오차) 수식은 `nn.MSELoss()`로 바꿉니다.  
최적화 도구에는 직접 만든 `[W, b]` 대신 `model.parameters()`를 전달합니다.


In [ ]:
# Lab 02_1: MSE 직접 계산
# loss = ((prediction - y_train) ** 2).mean()

# (변경) MSE 손실함수
loss_function = nn.MSELoss()


# Lab 02_1: W·b 직접 등록
# optimizer = torch.optim.SGD([W, b], lr=0.01)

# (변경) 모델 매개변수 등록
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)


### 6. 한 번의 학습 흐름 확인하기

학습 순서는 바뀌지 않습니다.

1. `model(x_train)` → 현재 모델로 예측
2. `loss_function(...)` → 예측 오차 계산
3. `zero_grad()` → 이전 기울기 초기화
4. `backward()` → 기울기 계산
5. `step()` → 가중치와 편향 갱신


In [ ]:
# Lab 02_1: 직접 순전파
# prediction = W * x_train + b

# (변경) 모델 순전파
prediction = model(x_train)

# Lab 02_1: MSE 직접 계산
# loss = ((prediction - y_train) ** 2).mean()

# (변경) 손실 계산
loss = loss_function(prediction, y_train)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("한 번 학습한 뒤의 손실:", loss.item())


### 7. 학습 반복하기

전체 학습 반복문의 구조는 `Lab 02_1`과 같습니다.  
핵심 변경은 예측과 손실 계산 두 줄입니다.


In [ ]:
# 비교 시작: 모델 초기화
torch.manual_seed(42)
model = LinearRegressionModel()
loss_function = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

epochs = 2000
loss_history = []

for epoch in range(epochs + 1):
    # Lab 02_1
    # prediction = W * x_train + b
    # loss = ((prediction - y_train) ** 2).mean()

    # (변경) 모델 순전파·손실 계산
    prediction = model(x_train)
    loss = loss_function(prediction, y_train)

    # 공통: 초기화 → 역전파 → 갱신
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if epoch % 400 == 0:
        weight = model.linear.weight.item()
        bias = model.linear.bias.item()
        print(
            f"epoch: {epoch:4d} | "
            f"loss: {loss.item():.8f} | "
            f"W: {weight:.4f} | b: {bias:.4f}"
        )


### 8. 학습 결과와 새 값 예측하기

학습 데이터는 $y=x$ 관계이므로 가중치는 1, 편향은 0에 가까워집니다.  
예측 단계에서는 기울기 계산이 필요하지 않으므로 `torch.no_grad()`를 사용합니다.


In [ ]:
print("학습된 W:", model.linear.weight.item())
print("학습된 b:", model.linear.bias.item())
print("최종 손실:", loss_history[-1])

x_new = torch.tensor([[5.0], [2.5], [1.5], [3.5]], dtype=torch.float32)

model.eval()
with torch.no_grad():
    y_new = model(x_new)

print("새 입력:", x_new.squeeze(dim=1))
print("예측 결과:", y_new.squeeze(dim=1))


## 확인 실습

입력 전압과 실제 측정값의 관계가 $y=20x+1$인 센서 보정 모델을 학습합니다.  
아래 TODO(투두, 작성할 부분)를 완성합니다.


In [ ]:
voltage_data = torch.tensor(
    [[0.5], [1.0], [1.5], [2.0]],
    dtype=torch.float32,
)
measured_data = torch.tensor(
    [[11.0], [21.0], [31.0], [41.0]],
    dtype=torch.float32,
)

# TODO 1: 모델 생성
sensor_model = LinearRegressionModel()

# TODO 2: 손실함수·최적화 도구
sensor_loss_function = nn.MSELoss()
sensor_optimizer = torch.optim.SGD(sensor_model.parameters(), lr=0.05)

# TODO 3: 학습
for epoch in range(3000):
    sensor_prediction = sensor_model(voltage_data)
    sensor_loss = sensor_loss_function(sensor_prediction, measured_data)

    sensor_optimizer.zero_grad()
    sensor_loss.backward()
    sensor_optimizer.step()

sensor_model.eval()
with torch.no_grad():
    result_at_1_25v = sensor_model(torch.tensor([[1.25]]))

print("센서 W:", sensor_model.linear.weight.item())
print("센서 b:", sensor_model.linear.bias.item())
print("1.25 V 보정값:", result_at_1_25v.item())


## 실행 확인

아래 셀에서 오류가 없으면 `Lab 02_2`가 정상적으로 완료된 것입니다.


In [ ]:
assert x_train.shape == torch.Size([3, 1])
assert y_train.shape == torch.Size([3, 1])
assert isinstance(model, nn.Module)
assert isinstance(model.linear, nn.Linear)
assert model.linear.in_features == 1
assert model.linear.out_features == 1
assert loss_history[-1] < 1e-4
assert abs(model.linear.weight.item() - 1.0) < 0.02
assert abs(model.linear.bias.item()) < 0.05
assert abs(result_at_1_25v.item() - 26.0) < 0.2

print("Lab 02_2 실행 확인 완료")


## 핵심 정리

- 모델: `W`, `b` 직접 생성 → `nn.Module`, `nn.Linear`
- 순전파: `W * x + b` → `model(x)`
- 손실함수: MSE 직접 계산 → `nn.MSELoss()`
- 학습 매개변수: `[W, b]` → `model.parameters()`
- 학습 반복: `zero_grad → backward → step` 구조 유지

## 다음 단계

`Lab 02_3`에서는 입력 특성을 여러 개로 확장한 다중 선형회귀를 구성합니다.  
ReLU는 비선형 관계를 학습하는 신경망 단계에서 추가합니다.
